# Naive Bayes / Sentiment Analysis Project

# Clasificación de reseñas de Google Play con Naive Bayes

## Objetivo del proyecto

En este proyecto voy a construir un clasificador de sentimientos para reseñas de Google Play. La idea es predecir si una reseña es positiva o negativa a partir del texto escrito por el usuario.

El enfoque seguirá una lógica simple y progresiva:

- cargar y revisar el dataset;
- entender qué variables aportan al problema;
- preparar el texto para el modelado;
- probar las tres variantes de Naive Bayes vistas en clase;
- comparar resultados;
- elegir la opción más razonable y documentar la decisión.

La intención no es complicar innecesariamente el pipeline, sino construir una solución clara, defendible y consistente con lo trabajado en el bootcamp.

## 1. Carga inicial del dataset

In [4]:
import pandas as pd

url = "https://raw.githubusercontent.com/4GeeksAcademy/naive-bayes-project-tutorial/main/playstore_reviews.csv"
df = pd.read_csv(url)

print(df.shape)
display(df.head())

print("\nInformación general del dataset:")
df.info()

(891, 3)


,package_name,review,polarity
0,com.facebook.katana,privacy at least put some option appear offline. i mean for some people like me it's a big pressure to be seen online like you need to response on every message or else you be called seenzone onl...,0
1,com.facebook.katana,"messenger issues ever since the last update, initial received messages don't get pushed to the messenger app and you don't get notification in the facebook app or messenger app. you open the face...",0
2,com.facebook.katana,profile any time my wife or anybody has more than one post and i view them it would take me to there profile so that i can view them all at once. now when i try to view them it tells me that the ...,0
3,com.facebook.katana,the new features suck for those of us who don't have a working back button can you guys make the videos able to be slid to the left to exit the video. as i have to force close facebook to exit,0
4,com.facebook.katana,forced reload on uploading pic on replying comment last night i tried to reply a comment by uploading a photo from my phone. when i press on the button to select photos the app automatically goes...,0



Información general del dataset:
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   package_name  891 non-null    str  
 1   review        891 non-null    str  
 2   polarity      891 non-null    int64
dtypes: int64(1), str(2)
memory usage: 21.0 KB


## 2. Revisión de variables y enfoque de modelado

Una vez cargado el dataset, el siguiente paso es revisar qué variables contiene y cuál de ellas aporta realmente al problema de clasificación.

En este caso, el objetivo es predecir si una reseña es positiva o negativa. Por eso, la variable más importante para el modelado es el texto de la reseña (`review`), ya que ahí está el contenido semántico que expresa el sentimiento del usuario.

La columna `package_name` puede aportar contexto sobre la aplicación, pero no describe directamente el tono de la opinión. Además, este proyecto busca construir un clasificador de texto sencillo y coherente con la lógica del módulo, así que el enfoque principal será trabajar con:

- `review` como variable predictora;
- `polarity` como variable objetivo.

### Decisión inicial

Después de revisar la estructura del dataset, el enfoque de trabajo será:

- conservar `review` como entrada principal del modelo;
- usar `polarity` como etiqueta binaria;
- dejar `package_name` fuera del modelado base, ya que para este caso añade menos valor que el propio contenido textual.

Esta decisión simplifica el pipeline y mantiene el proyecto alineado con un problema clásico de análisis de sentimiento.

### Notas de revisión de variables

- Número de filas y columnas: **891 filas y 3 columnas**
- Variables disponibles: **`package_name`, `review` y `polarity`**
- Valores nulos por columna: **no se detectaron valores nulos en ninguna de las tres variables**
- Distribución de la variable objetivo: **la clase `0` representa aproximadamente el 65.54% del dataset (584 registros) y la clase `1` el 34.46% (307 registros)**
- Observación inicial: **el dataset está limpio a nivel estructural, con reseñas en texto libre y una variable objetivo binaria ya definida. Aunque `package_name` identifica la aplicación, la información más útil para clasificar el sentimiento está en `review`. Además, la variable objetivo presenta cierto desbalance, por lo que conviene tenerlo en cuenta al comparar modelos.**

In [5]:
# Revisión general de estructura
print("Shape del dataset:", df.shape)
print("\nColumnas:")
print(df.columns.tolist())

print("\nInformación general:")
df.info()

# Valores nulos
print("\nValores nulos por columna:")
print(df.isnull().sum())

# Número de valores únicos por columna
print("\nValores únicos por columna:")
print(df.nunique())

# Distribución de la variable objetivo
print("\nDistribución de 'polarity' (conteo):")
print(df["polarity"].value_counts())

print("\nDistribución de 'polarity' (proporción):")
print(df["polarity"].value_counts(normalize=True))

# Muestra de observaciones
print("\nMuestra aleatoria de registros:")
display(df.sample(5, random_state=42))

# Algunas reseñas para inspección manual
print("\nEjemplos de reseñas:")
for i, text in enumerate(df["review"].dropna().head(3), start=1):
    print(f"\nReseña {i}:")
    print(text[:300], "...")

Shape del dataset: (891, 3)

Columnas:
['package_name', 'review', 'polarity']

Información general:
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   package_name  891 non-null    str  
 1   review        891 non-null    str  
 2   polarity      891 non-null    int64
dtypes: int64(1), str(2)
memory usage: 21.0 KB

Valores nulos por columna:
package_name    0
review          0
polarity        0
dtype: int64

Valores únicos por columna:
package_name     23
review          891
polarity          2
dtype: int64

Distribución de 'polarity' (conteo):
polarity
0    584
1    307
Name: count, dtype: int64

Distribución de 'polarity' (proporción):
polarity
0    0.655443
1    0.344557
Name: proportion, dtype: float64

Muestra aleatoria de registros:


,package_name,review,polarity
709,com.opera.mini.native,love/hate has bug and security issues. i tried to report that facebook and google plus have security issues and it wouldn't allow me to do so! well i just did didn't i! ......,0
439,com.whatsapp,whatsapp i use this app now that blackberry messenger has basically gone away. my friends & family live all over the world. this really helps keep us in touch!,1
840,com.hamropatro,usefully verry nice app,1
720,com.opera.mini.native,fonts why in the heck is this thing analysing my fonts??? not really quick browsing when i have to wait 5minutes for the fonts to load. are you asking my opinion? avoid this. terrible,0
39,com.facebook.katana,app doesn't work after latest upgrade the facebook app refuses to work on my mobile data (3g) after the latest upgrade! it says it cannot connect right now.,0



Ejemplos de reseñas:

Reseña 1:
 privacy at least put some option appear offline. i mean for some people like me it's a big pressure to be seen online like you need to response on every message or else you be called seenzone only. if only i wanna do on facebook is to read on my newsfeed and just wanna response on message i want to ...

Reseña 2:
 messenger issues ever since the last update, initial received messages don't get pushed to the messenger app and you don't get notification in the facebook app or messenger app. you open the facebook app and happen to see you have a message. you have to click the icon and it opens messenger. subseq ...

Reseña 3:
 profile any time my wife or anybody has more than one post and i view them it would take me to there profile so that i can view them all at once. now when i try to view them it tells me that the page that i requested is not available. i've restarted my phone i even cleard the cache and i've uninsta ...


## 3. Preparación básica del texto

Después de revisar las variables, preparo una versión del dataset enfocada únicamente en el problema de clasificación de sentimiento.

En esta etapa conservo:

- `review` como texto de entrada;
- `polarity` como variable objetivo.

La limpieza aplicada aquí es intencionalmente simple. No busco transformar demasiado el texto, sino dejarlo en un formato consistente para la vectorización posterior. En concreto:

- convierto el texto a minúsculas;
- elimino espacios sobrantes al inicio y al final;
- normalizo espacios múltiples dentro de la reseña.

Dado que se trata de un pipeline base de clasificación de texto, prefiero evitar una limpieza excesiva antes de probar el modelo.

### Decisión de preprocesamiento

En este punto dejo fuera la variable `package_name` del modelado base. Aunque puede aportar contexto sobre la aplicación, el objetivo del proyecto es clasificar el sentimiento a partir del contenido de la reseña.

Por eso, el dataset de trabajo se construye solo con:

- `review_clean`: versión limpiada del texto;
- `polarity`: etiqueta binaria.

Antes de continuar, también verifico que la limpieza no haya generado textos vacíos.

### Observaciones de limpieza

- Columnas conservadas para modelado: **`review`, `polarity` y `review_clean`**
- Reseñas vacías después de limpiar: **0**
- Ejemplo antes/después de limpieza: **la limpieza eliminó espacios innecesarios al inicio y normalizó el formato del texto, manteniendo el contenido original. Por ejemplo, una reseña que empezaba con un espacio inicial pasó a una versión equivalente pero más consistente para el modelado.**
- Observación inicial: **la limpieza aplicada no alteró el contenido semántico de las reseñas y permitió generar una columna de texto lista para el siguiente paso. Además, no se produjeron textos vacíos, por lo que el dataset puede usarse directamente para hacer la partición entre entrenamiento y prueba.**

In [6]:
# Dataset de trabajo para modelado
df_ml = df[["review", "polarity"]].copy()

# Limpieza básica del texto
df_ml["review_clean"] = (
    df_ml["review"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# Verificación de reseñas vacías después de limpiar
empty_reviews = (df_ml["review_clean"] == "").sum()

print("Columnas del dataset de modelado:")
print(df_ml.columns.tolist())

print("\nShape del dataset de modelado:")
print(df_ml.shape)

print("\nReseñas vacías después de limpiar:")
print(empty_reviews)

print("\nPrimeras filas del dataset de modelado:")
display(df_ml.head())

print("\nEjemplos antes y después de limpieza:")
sample_examples = df_ml[["review", "review_clean"]].head(3)

for i, row in sample_examples.iterrows():
    print(f"\nEjemplo {i}")
    print("Original:")
    print(row["review"][:250], "...")
    print("\nLimpio:")
    print(row["review_clean"][:250], "...")

Columnas del dataset de modelado:
['review', 'polarity', 'review_clean']

Shape del dataset de modelado:
(891, 3)

Reseñas vacías después de limpiar:
0

Primeras filas del dataset de modelado:


,review,polarity,review_clean
0,privacy at least put some option appear offline. i mean for some people like me it's a big pressure to be seen online like you need to response on every message or else you be called seenzone onl...,0,privacy at least put some option appear offline. i mean for some people like me it's a big pressure to be seen online like you need to response on every message or else you be called seenzone only...
1,"messenger issues ever since the last update, initial received messages don't get pushed to the messenger app and you don't get notification in the facebook app or messenger app. you open the face...",0,"messenger issues ever since the last update, initial received messages don't get pushed to the messenger app and you don't get notification in the facebook app or messenger app. you open the faceb..."
2,profile any time my wife or anybody has more than one post and i view them it would take me to there profile so that i can view them all at once. now when i try to view them it tells me that the ...,0,profile any time my wife or anybody has more than one post and i view them it would take me to there profile so that i can view them all at once. now when i try to view them it tells me that the p...
3,the new features suck for those of us who don't have a working back button can you guys make the videos able to be slid to the left to exit the video. as i have to force close facebook to exit,0,the new features suck for those of us who don't have a working back button can you guys make the videos able to be slid to the left to exit the video. as i have to force close facebook to exit
4,forced reload on uploading pic on replying comment last night i tried to reply a comment by uploading a photo from my phone. when i press on the button to select photos the app automatically goes...,0,forced reload on uploading pic on replying comment last night i tried to reply a comment by uploading a photo from my phone. when i press on the button to select photos the app automatically goes ...



Ejemplos antes y después de limpieza:

Ejemplo 0
Original:
 privacy at least put some option appear offline. i mean for some people like me it's a big pressure to be seen online like you need to response on every message or else you be called seenzone only. if only i wanna do on facebook is to read on my new ...

Limpio:
privacy at least put some option appear offline. i mean for some people like me it's a big pressure to be seen online like you need to response on every message or else you be called seenzone only. if only i wanna do on facebook is to read on my news ...

Ejemplo 1
Original:
 messenger issues ever since the last update, initial received messages don't get pushed to the messenger app and you don't get notification in the facebook app or messenger app. you open the facebook app and happen to see you have a message. you hav ...

Limpio:
messenger issues ever since the last update, initial received messages don't get pushed to the messenger app and you don't get notificat

## 4. Separación entre entrenamiento y prueba

Antes de vectorizar el texto, separo el dataset en conjuntos de entrenamiento y prueba.

Este orden es importante porque el vectorizador debe ajustarse únicamente con los datos de entrenamiento y luego aplicarse al conjunto de prueba. De esta forma, la evaluación del modelo se mantiene más realista y evita fuga de información.

En este caso, usaré:

- `review_clean` como variable predictora;
- `polarity` como variable objetivo;
- una partición de entrenamiento y prueba con estratificación para conservar la proporción de clases.

### Criterio de partición

La variable objetivo tiene un cierto desbalance, por lo que conviene usar `stratify` en el `train_test_split`. Así, tanto el conjunto de entrenamiento como el de prueba mantienen una distribución de clases parecida a la del dataset original.

Este paso deja preparado el pipeline para la vectorización con `CountVectorizer` en la siguiente sección.

### Observaciones del split

- Tamaño de `X_train`: **712 registros**
- Tamaño de `X_test`: **179 registros**
- Tamaño de `y_train`: **712 registros**
- Tamaño de `y_test`: **179 registros**
- Distribución de clases en entrenamiento: **la clase `0` tiene 467 registros (65.59%) y la clase `1` tiene 245 registros (34.41%)**
- Distribución de clases en prueba: **la clase `0` tiene 117 registros (65.36%) y la clase `1` tiene 62 registros (34.64%)**
- Observación inicial: **la partición se realizó correctamente y conservó de forma muy similar la proporción de clases del dataset original gracias al uso de `stratify`. Esto permite continuar con la vectorización sin introducir un sesgo adicional entre entrenamiento y prueba.**

In [7]:
from sklearn.model_selection import train_test_split

X = df_ml["review_clean"]
y = df_ml["polarity"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Tamaño de X_train:", X_train.shape)
print("Tamaño de X_test:", X_test.shape)
print("Tamaño de y_train:", y_train.shape)
print("Tamaño de y_test:", y_test.shape)

print("\nDistribución de clases en y_train (conteo):")
print(y_train.value_counts())

print("\nDistribución de clases en y_train (proporción):")
print(y_train.value_counts(normalize=True))

print("\nDistribución de clases en y_test (conteo):")
print(y_test.value_counts())

print("\nDistribución de clases en y_test (proporción):")
print(y_test.value_counts(normalize=True))

print("\nEjemplo de reseña en entrenamiento:")
print(X_train.iloc[0][:300], "...")

print("\nEtiqueta asociada:")
print(y_train.iloc[0])

Tamaño de X_train: (712,)
Tamaño de X_test: (179,)
Tamaño de y_train: (712,)
Tamaño de y_test: (179,)

Distribución de clases en y_train (conteo):
polarity
0    467
1    245
Name: count, dtype: int64

Distribución de clases en y_train (proporción):
polarity
0    0.655899
1    0.344101
Name: proportion, dtype: float64

Distribución de clases en y_test (conteo):
polarity
0    117
1     62
Name: count, dtype: int64

Distribución de clases en y_test (proporción):
polarity
0    0.653631
1    0.346369
Name: proportion, dtype: float64

Ejemplo de reseña en entrenamiento:
best app i can't believe that it is free! they treat it as if you paid for it. ...

Etiqueta asociada:
1
